# BrightLearn Machine Learning Research
## Credit Risk: Predicting Loan Default

**Research question:** To what extent can supervised machine learning predict whether a borrower is likely to default on a loan using financial and credit-related information?

**Dataset:** Credit Risk Dataset (Lao Tse, Kaggle), 32,581 records and 12 columns.

**Target variable:** loan_status (0 = non-default; 1 = default).

> Educational exercise only. This dataset is described as simulated credit-bureau data. The model must not be used to approve or reject real loan applicants.

## 1. Research aim and machine-learning problem

The aim of this notebook is to explore whether a machine-learning model can learn patterns in borrower and loan information that help predict loan default.

This is **supervised learning** because the dataset already contains the outcome (loan_status) for each record. Additionaly, it is a **binary classification** task because the target has two classes or possible outcomes: 0 and 1, O = the borrower did not default (False) and 1 = the borrower defaulted (True)

I will:
1. Load and inspect the CSV file.
2. Check data quality, missing values, duplicates and unusual values.
3. Explore the target distributions.
4. Separate input features (X) from the target/output (y).
5. Split the data into training and testing sets before fitting preprocessing steps such as encoding, scaling and imputation.
6. Impute missing values, encode categorical features and scale numeric features.
7. Train classification models.
8. Evaluate models, paying attention to the default class (1).

## 2. Import libraries

In [0]:
# Install Seaborn
!pip install seaborn

dbutils.library.restartPython() 

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Import Scikit-Learn tools
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 3. Data Ingestion

I downloaded my dataset "credit_risk_dataset.csv" from the Kaggle dataset page.

Kaggle dataset page: https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data

In [0]:
# Read and convert spark table to pandas dataframe 
df = spark.table("credit_risk.data.prediction").toPandas()

# See first five records
display(df.head())



## 4. Data Inspection
The expected original dataset has about **32,581 rows and 12 columns**. 

In [0]:
# Confirming data shape 
df.shape

# Dataset has 32581 rows and 12 columns

In [0]:
# Check names of all columns by list
df.columns.tolist()

In [0]:
# Check data types and non-null counts 
df.info()

## Observations:
 # 4 categorical columns and 8 numerical columns 
 # All features have an appropriate data type 
 # person_emp_length has missing values available= 31686/32581, missing = 895 
 # loan_int_rate has missing values available = 29465/32581, missing = 3116

In [0]:
# Check statistical summary of numeric columns
df.describe().T

In [0]:
# Display last 5 records
df.tail()

####### Inspect missing values, duplicates, and standardise column names
- During basic data inspection (df.info()), I have identified up that two columns have missing values out of the 12:
> - `person_emp_length` has 895 missing values and,
> - `loan_int_rate` has 3116 missing values
- I will impute missing feature values later inside a pipeline, rather than deleting every row as missing values are not automatically errors
- The published dataset has consistent column names and correct spellings


In [0]:
# Verifying missing values 
# Count missing values in every column
missing_values = df.isnull().sum()

display(missing_values)

In [0]:
# Calculate percentage of missing values in dataset 
missing_percentage = (df.isnull().mean() * 100).round(2)

display(missing_percentage)

In [0]:
# Show duplicated records using f-string
number_of_duplicates = df.duplicated().sum()
print(f"The number of duplicates is {number_of_duplicates}")



In [0]:
# Display all occurrences of exact duplicate rows
duplicates = df[df.duplicated(keep=False)]

display(duplicates)

print("Exact duplicate rows:", df.duplicated().sum())

In [0]:
# Check how many rows remain after removing exact duplicates 
df_unique = df.drop_duplicates()

print("Original rows:", len(df))
print("Rows after removing exact duplicates:", len(df_unique))
print("Rows removed:", len(df) - len(df_unique))

In [0]:
# Replace our working dataset with the deduplicated dataset
df = df_unique.copy()

# Verify the result
print("Updated dataset shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

######### Duplicates
- The dataset initially contained 32,581 records.
- A duplicate check using Pandas identified 165 exact duplicate rows across all 12 columns. 
- These repeated records were removed using the drop_duplicates() function, retaining the first occurrence of each record. This resulted in 32,416 records. 
- Since the dataset does not contain a unique borrower identifier, it cannot be established whether identical records represent the same individual or separate individuals with identical recorded attributes.

####### Check data types and basic data quality
- The dataset contains both numeric and categorical features.
- I will check the unique values in the target and inspect categorical values. 

In [0]:
# Identify categorical columns
categorical_columns = df.select_dtypes(include="object").columns

# Display the unique categories in each column
for col in categorical_columns:
    print(f"\nCategories in {col}:")
    print(df[col].unique())

In [0]:
# Understanding target value
print("Target values:")
print(df["loan_status"].value_counts(dropna=False))

## 25,327 loan status confirm borrowers that did not default 
## 7,089 loan status confirms borrowers that defaulted 

In [0]:
## Calculating percentage of default status

# Count each loan status
status_counts = df["loan_status"].value_counts()

# Calculate the percentage of each class
status_percentage = df["loan_status"].value_counts(normalize=True) * 100

# Combine counts and percentages into one table
target_distribution = pd.DataFrame({
    "Number of records": status_counts,
    "Percentage (%)": status_percentage.round(2)
})

display(target_distribution)

########## Loan_status observations:
- The target variable, `loan_status`, showed an imbalance between the two classes. 
- Non-default records accounted for 78.13% of the dataset, while default records represented 21.87%. 
> - This indicates that the dataset contains considerably more non-default cases. 
> - Therefore, model evaluation should consider precision, recall and F1-score in addition to accuracy to ensure that the minority default class is adequately identified.

####### Validate the target
- `loan_status` is the answer the model is trying to predict. A record without a target cannot be used for supervised training, so we remove only rows where the target is missing.
- I will convert the target to integer and check that its values are 0 and 1. 

In [0]:
# Remove records where the target variable is missing
df = df.dropna(subset=["loan_status"]).copy()

# Validate that the target is numerical if can't be changed, replace with NaN
df["loan_status"] = pd.to_numeric(df["loan_status"], errors="coerce")

# Remove any missing target values created during conversion.
df = df.dropna(subset=["loan_status"]).copy()

# Convert the target into whole numbers (integers).
df["loan_status"] = df["loan_status"].astype(int)

# Display target classes after validation and updated shape, check the unique target values and sort them
print("Target classes after validation:", sorted(df["loan_status"].unique()))
print("Updated shape:", df.shape)

## target values are 0 and 1 as expected to represent defaul and non-default

## 5. Exploratory data analysis (EDA)
EDA helps us understand the dataset before modelling.

### 5.1 Target distribution

The target distribution shows how many loans are labelled non-default (0) and default (1). The non-default class(0) is much more common than the default class (1), the dataset is imbalanced. In this situation, accuracy alone can be misleading.

In [0]:
# Re-calculating the target distribution 
target_counts = df["loan_status"].value_counts().sort_index()

target_percent = (
    df["loan_status"].value_counts(normalize=True).sort_index() * 100
)

display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_percent.round(2)
}))


In [0]:
# Visualizing target distribution in bar chart
plt.figure(figsize=(6, 4))

sns.countplot(data=df, x="loan_status")

plt.title("Distribution of Loan Status")
plt.xlabel("Loan status (0 = non-default, 1 = default)")
plt.ylabel("Number of records")

plt.show()

####### 8.2 Numeric feature distributions
- I will inspect all numeric columns, excluding the target value `loan_status`.

In [0]:
# Select all numerical columns in the dataset
numeric_columns_all = df.select_dtypes(include=np.number).columns.tolist()

# Remove loan_status because it is our target variable, only analyse numerical input features
numeric_features_eda = [
    c for c in numeric_columns_all if c != "loan_status"
]

# Create histograms for all numerical input features
df[numeric_features_eda].hist(
    figsize=(16, 12),
    bins=30, # bins=30 divides the data into 30 intervals
    edgecolor="black"
)

# Add one main title above all the histograms
plt.suptitle("Distributions of Numeric Predictors", y=1.02)

# Automatically adjust spacing so charts do not overlap
plt.tight_layout()

# Display all the histograms
plt.show()



### 5.3 Categorical feature distributions

In [0]:
# Identify all categorical columns in our dataset
categorical_features_eda = df.select_dtypes(
    include=["object"]
).columns.tolist()

# Loop through each categorical column
for col in categorical_features_eda:

    # Create a new chart for the current column
    plt.figure(figsize=(9, 4))

    # Arrange categories from most frequent to least frequent
    order = df[col].value_counts().index

    # Create a bar chart showing the frequency of each category
    sns.countplot(data=df, x=col, order=order)

    # Add a title to the chart
    plt.title(f"Count of {col}")

    # Rotate category labels so they are easier to read
    plt.xticks(rotation=35, ha="right")

    # Adjust chart spacing
    plt.tight_layout()

    # Display the chart
    plt.show()

######### Chart Interpretation
Chart 1: Count of home ownership shows: 
- RENT is the most common category
- MORTGAGE is the second most common
- OWN is considerably less common
- OTHER has very few records

Interpretation: Most borrowers in this dataset are renting or have a mortgage rather than owning their homes outright.

Chart 2: Count of loan intention shows:
- EDUCATION	is Most frequent
- MEDICAL is Second most frequent
- VENTURE is Relatively common
- PERSONAL is Relatively common
- DEBTCONSOLIDATION	is Relatively common
- HOMEIMPROVEMENT is Least frequent

Interpretation: Education and medical loans appear frequently, while home improvement loans are less common.

Chart 3: Count of loan grade shows: 
- A and B are the most common grades
- C is also relatively common
- D appears less frequently
- E and F have relatively few records
- G is extremely rare

Interpretation: Most records fall within grades A, B and C.

Chart 4: Count of previous person default file 
- N is significantly more common.
- Y is considerably less common.

This means most borrowers do not have a previous default recorded in this particular feature.


### 5.4 Outlier inspection using boxplots
A boxplot helps identify values that are unusually far from the rest of a numeric distribution. A point outside the whiskers is a potential outlier, not automatically a mistake.
- I will not automatically delete outliers because high incomes or large loans may be genuine observations.

In [0]:
# Loop through every numerical predictor
for col in numeric_features_eda:
    plt.figure(figsize=(8, 3))

# Draw a boxplot for the selected numerical column
    sns.boxplot(x=df[col])

    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

In [0]:
# Verifying outliers in min and max values
df.describe()

## Unusual values verified on max for all input features
 # Age values extend beyond 120 and even towards 144 - Ages above a realistic human lifespan
 # Employment length values extend beyond 120 years - Employment lengths are above a realistic working lifetime
 # Income- some extremely high values, including around 6 million - Check whether these incomes are genuine or data errors

######## Investigate outliers using IQR

In [0]:
# Create an empty list to store our outlier results
outlier_results = []

# Loop through every numerical feature
for col in numeric_features_eda:

    # Calculate the first quartile (25th percentile)
    Q1 = df[col].quantile(0.25)

    # Calculate the third quartile (75th percentile)
    Q3 = df[col].quantile(0.75)

    # Calculate the interquartile range
    IQR = Q3 - Q1

    # Calculate the lower boundary
    lower_bound = Q1 - (1.5 * IQR)

    # Calculate the upper boundary
    upper_bound = Q3 + (1.5 * IQR)

    # Count values below or above the boundaries
    outlier_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    # Store the results for this column
    outlier_results.append({
        "Feature": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outlier_count
    })

# Convert the results into a DataFrame for easy viewing
outlier_summary = pd.DataFrame(outlier_results)

# Display the results
display(outlier_summary)

In [0]:
## inspect the highest and lowest values in the columns

# Select the numerical columns we want to investigate
columns_to_check = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

# Display the lowest and highest values in each column
for col in columns_to_check:

    print(f"\nChecking: {col}")

    print("Lowest values:")
    print(df[col].nsmallest(5).tolist())

    print("Highest values:")
    print(df[col].nlargest(10).tolist())

####### Results of Features, highest values found and potential vlaues
- person_age - 144, 144, 144, 123, 123 - **Investigate** - These ages are not realistic for ordinary borrower records.
- person_income - 6,000,000; 2,039,784; 1,900,000 - **Investigate** - Unusually high, but potentially valid.
- person_emp_length	- 123, 123, 41, 38, 34 - **Investigate** - 123 years is not realistic employment history.
- loan_amnt	- 35,000 - Potentially valid.
- loan_percent_income - 0.83, 0.78, 0.77 - Unusually high borrowing relative to income, but potentially valid.
- cb_person_cred_hist_length - 30 - Potentially valid.

In [0]:
# Count impossible or highly suspicious values
print("Age above 100:", (df["person_age"] > 100).sum())

print("Employment above 50 years:",
      (df["person_emp_length"] > 50).sum())

## The records below will be treated as data quality concern results

## 7. Prepare predictors (X) and target (y)
- `X` contains the information used to make predictions = Information about the borrower and loan, such as age, income, loan amount and employment length.
- `y` contains the correct outcome the model must learn to predict = whether the borrower defaulted


In [0]:
# Identify the target
target_column = "loan_status"

# Define X and y
X = df.drop(columns=[target_column]).copy()
y = df[target_column].copy()

# Check the structure of X and y
print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [0]:
# Display first 5 feature records
display(X.head())

# Display first 5 target values
display(y.head())

## Observation:
 # A person aged 22 is said to have an employment length of 123 years which is quite impossible

In [0]:

# Remove records with clearly impossible ages or employment lengths, that is 5 records with ages> 100, and 2 records with employment legnth> 50 years
df = df[
    (df["person_age"] <= 100) &
    (df["person_emp_length"] <= 50)
].copy()

# Check the updated dataset size
print("Updated dataset shape:", df.shape)

## Updated dataset = 31,522 rows and 12 columns

In [0]:
# Separate features and target again
X = df.drop(columns=["loan_status"]).copy()
y = df["loan_status"].copy()

## 8. Identify numeric and categorical predictors
- Identifying them from the data types rather than manually guessing. The target is already separate.

In [0]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 9. Split into training and testing sets (80/20)


In [0]:
# Import the train-test splitting function
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the resulting dataset sizes
print("Training predictors:", X_train.shape)
print("Testing predictors:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

######### Observation: Before splitting, the earlier screenshot showed 32,416 records. The split now accounts for 31,522, a difference of 894. This may be because the age and employment-length filtering also excluded records with missing values in those columns.

In [0]:
# Verifying the number of records remaining after cleaning
print("Current cleaned dataset:", df.shape)

# Check missing values in the two columns used for filtering
print(df[["person_age", "person_emp_length"]].isnull().sum())

## 10. Preprocess the features
- Preparing raw data so a machine-learning algorithm can use it.

I'll build two separate workflows:

**Numeric columns**
- Replace missing values with the median.
- Standardise values using `StandardScaler`.

**Categorical columns**
- Replace missing values with the most frequent category.
- Apply one-hot encoding to convert categories into numeric indicator columns.

In [0]:
# 10.1 Identify numerical and categorical columns 

# Select numerical columns
numeric_features = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

# Select categorical columns
categorical_features = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

In [0]:
# 10.2 Import Scikit-learn preprocessing tools

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [0]:
# 10.3 Create numerical flow

# Fill missing numerical values using the median,
# then standardise the numerical columns.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [0]:
# 10.4 Create a categorical flow

# Fill missing categories using the most frequent value (mode),
# then convert categories into numerical columns.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [0]:
# 10.5 Combine both workflows

# Apply the numerical workflow to numerical columns
# and the categorical workflow to categorical columns.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [0]:
# 10.6 Fit on training data and transform on both sets

#  Fit = Learn the imputation values, scaling parameters and categories
# from the training data only.
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the already-learned transformations to the test data.
X_test_processed = preprocessor.transform(X_test)

# Display the resulting dimensions
print("Processed training data:", X_train_processed.shape)
print("Processed testing data:", X_test_processed.shape)

## Both datasets have been transformed succesfully, see cell 53 (split train_test)

In [0]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing steps created.")

## 11. Create a baseline model

A baseline is a simple point of comparison. `DummyClassifier` predicts the most frequent class every time.

In [0]:
# 11.1 Build baseline model

from sklearn.dummy import DummyClassifier

# Create a simple baseline model
baseline = DummyClassifier(strategy="most_frequent")

# Train the baseline using training data
baseline.fit(X_train_processed, y_train)

# Predict outcomes for the test data
baseline_predictions = baseline.predict(X_test_processed)

# Check baseline accuracy
from sklearn.metrics import accuracy_score

print("Baseline accuracy:",
      accuracy_score(y_test, baseline_predictions))

In [0]:
# 11.2 Train three classification models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Create the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# Train each model and generate predictions
predictions = {}

for name, model in models.items():

    # Train the model
    model.fit(X_train_processed, y_train)

    # Predict on unseen test data
    predictions[name] = model.predict(X_test_processed)

    print(name, "trained successfully")

In [0]:
# 11.3 Evaluate the models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Compare the performance of each model
for name, y_pred in predictions.items():

    print("\n", name)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))
    print("Recall:", recall_score(y_test, y_pred, zero_division=0))
    print("F1-score:", f1_score(y_test, y_pred, zero_division=0))

######## Results of three models
1. Logistic Regression
- Correctly classified approximately 87% of test records.
- Its recall of 55.95% is relatively low.
- It misses quite a number of actual class 1 cases.
2. Decision Tree
- Improved on Logistic Regression in accuracy, recall and F1-score.
- Its recall is 76.28%, meaning it identifies more of the actual class 1 cases.
3. Random Forest
- Achieved the highest accuracy at 93.70%.
- Its precision is very high at 97.63%.
- Its F1-score is also the highest at 83.28%.
- However, its recall is 72.61%, so it still misses some actual class 1 cases.

Random Forest is currently our strongest model overall. However, we should not select it based on accuracy alone.

My baseline accuracy was 78.40%.

All three models performed better than the baseline:
- Logistic Regression improved accuracy by approximately 8.50 percentage points.
- Decision Tree improved it by approximately 10.34 percentage points.
- Random Forest improved it by approximately 15.30 percentage points.

### 12. Create Confusion metrices for each model

In [0]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Create a confusion matrix for each model
for name, y_pred in predictions.items():

    # Compare actual values with predicted values
    cm = confusion_matrix(y_test, y_pred)

    # Display the confusion matrix
    display_matrix = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[0, 1]
    )

    display_matrix.plot(cmap="Blues")

    plt.title(f"Confusion Matrix - {name}")
    plt.show()

In [0]:
## Check if target values are still present

# Count the actual target classes
print(y.value_counts())

# Show their percentages
print(y.value_counts(normalize=True) * 100) 

## target values are still present, but still show imbalance 

##########Model evaluation and interpretation

The Random Forest model achieved the highest overall accuracy of 93.70% and an F1-score of 83.28%. It also recorded a precision of 97.63%, indicating that most borrowers it classified as likely to default were actual defaulters. However, its recall was 72.61%, meaning that it failed to identify some borrowers who eventually defaulted.

The Decision Tree model achieved the highest recall of 76.28%, correctly identifying a larger proportion of actual loan defaults. This is important in credit risk assessment because failing to identify a potential defaulter may expose a lender to financial losses. However, the Decision Tree also produced more false-positive predictions.

Overall, Random Forest demonstrated the strongest general performance based on the F1-score, while Decision Tree showed an advantage in identifying loan defaults. Therefore, model selection should consider the financial consequences of different prediction errors rather than accuracy alone.

### 13. Cross validation

In [0]:

# 13.1 Import the required libraries

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd



In [0]:
# 2. Create 5 stratified folds

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [0]:
# 3. Store the cross-validation results
cv_results = []

In [0]:
# 13.4. Evaluate each model
for name, model in models.items():

    # Combine preprocessing and the model
    full_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    # Perform cross-validation
    scores = cross_validate(
        full_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1"],
        n_jobs=-1
    )

    # Store the average scores
    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-score": scores["test_f1"].mean()
    })

In [0]:
# 13.5 Display the comparison
cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(4))

In [0]:
# Display previously coded test-set results


# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Store test-set results
test_results = []

# Evaluate each model's predictions
for name, y_pred in predictions.items():

    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    })

# Create and display the results table
test_results_df = pd.DataFrame(test_results)

display(test_results_df.round(4))


In [0]:
## Compare test-set results and cross-validation results

# Merge test-set and cross-validation results
comparison_df = test_results_df.merge(
    cv_results_df,
    on="Model",
    suffixes=("_Test", "_CV")
)

# Display the comparison
display(comparison_df.round(4))


In [0]:
# Calculate the difference between test and CV scores
comparison_df["F1_Difference"] = abs(
    comparison_df["F1-score_Test"] -
    comparison_df["F1-score_CV"]
)

display(
    comparison_df[
        ["Model", "F1-score_Test", "F1-score_CV", "F1_Difference"]
    ].round(4)
)


######### Explanation of difference
- Logistic Regression: Difference of 0.0078
- Decision Tree: Difference of 0.0007
- Random Forest: Difference of 0.0107

All three models show relatively small differences between the two evaluation approaches.

# 14. Cross-validation rreport
### Cross-validation and model reliability

- Five-fold stratified cross-validation was performed to assess the consistency of the three classification models. Stratified sampling was used to maintain the proportion of loan defaults and non-defaults across the five folds.

- The Random Forest model achieved the highest average accuracy of 93.31%, precision of 96.42% and F1-score of 82.21%. The Decision Tree model achieved the highest recall of 76.27%, indicating that it identified the largest proportion of actual loan defaults. Logistic Regression recorded the lowest overall performance, with an F1-score of 64.07%.

- The cross-validation results were also compared with the test-set results. The small differences between the F1-scores suggest that the models performed consistently across the different data splits. Random Forest remained the strongest overall model, while Decision Tree demonstrated an advantage in identifying default cases.

- Based on these findings, Random Forest was selected as the strongest overall model. However, further evaluation would be necessary before applying either model to real credit-risk decisions.

# 15. Conclusion

## Conclusion

This research explored the application of Machine Learning in predicting loan defaults using a credit risk dataset. The purpose was to understand how machine learning can use historical borrower information to identify patterns and predict whether a borrower is likely to default on a loan. The research also examined the different types of machine learning, data preparation techniques, classification algorithms and model evaluation methods.

The practical implementation was conducted using Python, Pandas and Scikit-learn in Databricks. The dataset contained 32,416 records after preparation, with 11 predictor variables and a target variable called `loan_status`. The target represented two outcomes: 0 for borrowers who did not default and 1 for borrowers who defaulted. Data preparation included investigating missing values, examining outliers, separating the predictors from the target, splitting the data into training and testing sets, and applying imputation, encoding and scaling through a preprocessing pipeline.

Three supervised classification algorithms were developed and evaluated: Logistic Regression, Decision Tree and Random Forest. Their performance was assessed using accuracy, precision, recall, F1-score, confusion matrices and five-fold stratified cross-validation. Random Forest achieved the strongest overall performance, with a test accuracy of 93.70% and an F1-score of 83.28%. Its cross-validation F1-score was 82.21%, suggesting relatively consistent performance across the validation folds. However, Decision Tree achieved the highest recall, indicating that it identified a larger proportion of actual loan defaults.

The findings demonstrate that machine learning can support credit risk analysis by identifying patterns associated with loan defaults. However, model accuracy alone is not sufficient to establish whether a model is suitable for real lending decisions. The consequences of false predictions, fairness, data quality and responsible use must also be considered.

Overall, this research provided practical experience in preparing data, developing classification models and evaluating their performance. It also demonstrated how machine learning can contribute to financial decision-making when implemented responsibly and supported by appropriate human oversight.

# 16. Limitations
## Limitations of the Research

Although the machine learning models achieved encouraging results, the research had several limitations.

Firstly, the dataset contained an imbalanced target variable. Approximately 78.41% of the records represented borrowers who did not default, while 21.59% represented borrowers who defaulted. This imbalance may influence model performance because algorithms can become more effective at predicting the majority class. Therefore, accuracy alone may not provide a complete picture of how well the models identify loan defaults. Precision, recall, F1-score and confusion matrices were also considered.

Secondly, the dataset represents a particular collection of borrower and loan records. Its patterns may not reflect the circumstances of all financial institutions, borrowers or economic environments. The models were trained and evaluated using this dataset, meaning their performance cannot automatically be generalised to other lending environments.

Thirdly, the quality and completeness of the available data may affect predictions. Missing values required imputation, while unusual numerical observations were investigated as potential outliers. Although preprocessing helps prepare data for modelling, it does not guarantee that every data quality issue has been eliminated.

Another limitation is that only three classification algorithms were compared: Logistic Regression, Decision Tree and Random Forest. Other algorithms and alternative model configurations could potentially produce different results. The research also used a particular train-test split and five-fold cross-validation, which provide useful evaluation evidence but cannot guarantee future real-world performance.

Finally, the dataset does not establish every factor that may influence a borrower's ability to repay a loan. Economic conditions, changes in employment, unexpected expenses and other circumstances may affect repayment behaviour but may not be fully represented in the available data.

These limitations mean that the findings should be interpreted within the scope of the dataset and research objectives. Further testing with additional data, different economic conditions and broader model comparisons would be necessary before considering real-world implementation.